In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers torch torchaudio requests

In [ ]:
!pip install evaluate

In [ ]:
import os
import evaluate
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, Audio
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
# 1. SETUP PATHS (Adjust 'dataset_path' to where your 'audio' folder is)
dataset_path = "/content/drive/MyDrive/Urban8KAudioFiles"
csv_path = "/content/drive/MyDrive/Urban8KAudioFiles/UrbanSound8K.csv"


In [ ]:

# 2. LOAD AND CLEAN METADATA
df = pd.read_csv(csv_path)

# Construct full paths: audio/foldX/filename.wav
df['file_path'] = df.apply(
    lambda row: os.path.join(dataset_path, f"fold{row['fold']}", row['slice_file_name']),
    axis=1
)

# Remove any rows with missing values in critical columns
df = df.dropna(subset=['file_path', 'classID'])



In [ ]:
# 3. VERIFY FILE EXISTENCE (Prevents NoneType errors later)
print("Verifying audio files on disk...")
df['exists'] = df['file_path'].apply(os.path.exists)
missing_count = len(df) - df['exists'].sum()

if missing_count > 0:
    print(f"⚠️ Warning: {missing_count} files missing. Removing them from training.")
    df = df[df['exists'] == True]



In [ ]:
# 4. CONVERT TO HUGGINGFACE DATASET
# Use Fold 10 for evaluation, Folds 1-9 for training
train_df = df[df['fold'] != 10][['file_path', 'classID']]
test_df = df[df['fold'] == 10][['file_path', 'classID']]

train_dataset = Dataset.from_pandas(train_df).rename_column("classID", "label")
test_dataset = Dataset.from_pandas(test_df).rename_column("classID", "label")

# Cast to Audio type (Resamples to 16kHz automatically)
train_dataset = train_dataset.cast_column("file_path", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("file_path", Audio(sampling_rate=16000))



In [ ]:
# 5. PREPROCESSING
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_id)

def preprocess_function(batch):
    # This assumes cast_column worked and we have a list of dicts with 'array'
    audio_arrays = [x["array"] for x in batch["file_path"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding="max_length",
        max_length=1024,
        return_tensors="pt"
    )
    inputs["labels"] = batch["label"]
    return inputs


In [10]:
# Map the dataset (remove_columns avoids ArrowInvalid length mismatch)
# 5.1 Encode Train Data
encoded_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=train_dataset.column_names
)
train_save_path = "/content/drive/MyDrive/Urban8KAudioFiles/test/processed_train"
encoded_train.save_to_disk(train_save_path)

Map:   0%|          | 0/7895 [00:00<?, ? examples/s]

In [ ]:
train_save_path = "/content/drive/MyDrive/Urban8KAudioFiles/test/processed_train"
encoded_train.save_to_disk(train_save_path)

In [ ]:
# 5.2 Encode Test Data
encoded_test = test_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=test_dataset.column_names
)
test_save_path = "/content/drive/MyDrive/Urban8KAudioFiles/test/processed_test"
encoded_test.save_to_disk(test_save_path)

In [ ]:
from datasets import load_from_disk
import torch

# 1. Load the processed datasets from your Drive
try:
    train_verify = load_from_disk("/content/drive/MyDrive/Urban8KAudioFiles/processed_train")
    test_verify = load_from_disk("/content/drive/MyDrive/Urban8KAudioFiles/processed_test")
    print(f"✅ Success! Loaded {len(train_verify)} training samples and {len(test_verify)} test samples.")
except Exception as e:
    print(f"❌ Error loading files: {e}")

# 2. Inspect a random sample (e.g., sample #100)
sample = train_verify[100]

print("\n--- Sample Verification ---")
print(f"Keys present in dataset: {sample.keys()}")

# Check the Spectrogram shape
# For AST, this should typically be [1, 1024, 128]
input_values = torch.tensor(sample['input_values'])
print(f"Spectrogram shape: {input_values.shape}")

# Check the Label
print(f"Label ID: {sample['labels']}")

# 3. Quick Data Validation
if input_values.abs().sum() > 0:
    print("✅ Data Integrity: Spectrogram contains non-zero numerical data.")
else:
    print("⚠️ Warning: Spectrogram appears to be empty (all zeros).")

✅ Success! Loaded 7895 training samples and 837 test samples.

--- Sample Verification ---
Keys present in dataset: dict_keys(['input_values', 'labels'])
Spectrogram shape: torch.Size([1024, 128])
Label ID: 8
✅ Data Integrity: Spectrogram contains non-zero numerical data.


In [ ]:
# 5.3 Saving the mapping into the drive.
train_save_path = "/content/drive/MyDrive/Urban8KAudioFiles/processed_train"
test_save_path = "/content/drive/MyDrive/Urban8KAudioFiles/processed_test"

if not os.path.exists(train_save_path):
    print("Mapping and saving to Drive...")
    encoded_train.save_to_disk(train_save_path)
    encoded_test.save_to_disk(test_save_path)
else:
    print("Loading processed data from Drive...")
    from datasets import load_from_disk
    encoded_train = load_from_disk(train_save_path)
    encoded_test = load_from_disk(test_save_path)

Mapping and saving to Drive...


Saving the dataset (0/9 shards):   0%|          | 0/7895 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/837 [00:00<?, ? examples/s]

In [ ]:
# 6. MODEL INITIALIZATION
model = ASTForAudioClassification.from_pretrained(
    model_id,
    num_labels=10,
    ignore_mismatched_sizes=True
)



In [ ]:
# 7. TRAINING ARGUMENTS
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Urban8KAudioFiles//ast_urban_results",
    eval_strategy="epoch",       # CHANGED from evaluation_strategy
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    optim="adamw_torch",
    warmup_steps=500,
    fp16=False,                  # Set to False because you are on XLA/TPU
    bf16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"             # Prevents errors if WandB isn't installed
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    compute_metrics=compute_metrics,
)



In [ ]:
# 8. START TRAINING
trainer.train()

In [ ]:
# Define the saving path on your Google Drive
save_directory = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"

# 1. Save the model
trainer.save_model(save_directory)

# 2. Save the feature extractor (very important for preprocessing during testing)
feature_extractor.save_pretrained(save_directory)

print(f"✅ Model and feature extractor saved to {save_directory}")

In [ ]:
test_path = df['file_path'].iloc[0]
print(f"Checking path: {test_path}")
print(f"Does it exist? {os.path.exists(test_path)}")

IndexError: single positional indexer is out-of-bounds